<a href="https://colab.research.google.com/github/ARPITAKAR/PyTorch_Light/blob/main/pytorch_nn_module.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
rjmanoj_credit_card_customer_churn_prediction_path = kagglehub.dataset_download('rjmanoj/credit-card-customer-churn-prediction')

print('Data source import complete.')


Using Colab cache for faster access to the 'credit-card-customer-churn-prediction' dataset.
Data source import complete.


In [19]:
!pip install kaggle

In [20]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle (1).json


{'kaggle (1).json': b'{"username":"arpit786","key":"90e84d7104af5105d6017a94d41b34e3"}'}

In [21]:
!mkdir -p ~/.kaggle

In [22]:
!cp kaggle.json ~/.kaggle/

In [23]:
!chmod 600 ~/.kaggle/kaggle.json

In [24]:
! kaggle datasets list

ref                                                                      title                                                     size  lastUpdated                 downloadCount  voteCount  usabilityRating  
-----------------------------------------------------------------------  --------------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
algozee/teenager-menthal-healy                                           Social Media Impact on Teen Mental Health                16190  2026-04-05 08:04:21.823000          30572        638                1  
laveshjadon/ai-impact-on-students                                        Impact of Ai on Students                               1187170  2026-05-10 23:12:10.070000           1693         46                1  
shambhurajejagadale/student-performance-prediction-dataset               Student Performance Prediction Dataset                   84282  2026-05-09 15:49:58.877000 

In [25]:
import kagglehub

path = kagglehub.dataset_download("rjmanoj/credit-card-customer-churn-prediction")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'credit-card-customer-churn-prediction' dataset.
Path to dataset files: /kaggle/input/credit-card-customer-churn-prediction


In [26]:
import os

print(os.listdir(path))

['Churn_Modelling.csv']


In [27]:
import pandas as pd
import os

csv_path = os.path.join(path, "Churn_Modelling.csv")

df = pd.read_csv(csv_path)

df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [28]:
df.drop(columns = ['RowNumber','CustomerId','Surname'],inplace=True)

In [29]:
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [30]:
df['Geography'].value_counts()

,count
Geography,
France,5014
Germany,2509
Spain,2477


In [31]:
df['Gender'].value_counts()

,count
Gender,
Male,5457
Female,4543


In [32]:
df = pd.get_dummies(df,columns=['Geography','Gender'],drop_first=True)

In [33]:
df.head()

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_Germany,Geography_Spain,Gender_Male
0,619,42,2,0.00,1,1,1,101348.88,1,False,False,False
1,608,41,1,83807.86,1,0,1,112542.58,0,False,True,False
2,502,42,8,159660.80,3,1,0,113931.57,1,False,False,False
3,699,39,1,0.00,2,0,0,93826.63,0,False,False,False
4,850,43,2,125510.82,1,1,1,79084.10,0,False,True,False


In [34]:
X = df.drop(columns=['Exited'])
y = df['Exited'].values

from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=0)

In [35]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

X_train_trf = scaler.fit_transform(X_train)
X_test_trf = scaler.transform(X_test)

# Numpy arrays to Pytorch Tensors

In [36]:
X_train_tensor = torch.from_numpy(X_train_trf).float()
X_test_tensor = torch.from_numpy(X_test_trf).float()
y_train_tensor = torch.from_numpy(y_train).float()
y_test_tensor = torch.from_numpy(y_test).float()

In [37]:
X_train_tensor.shape

torch.Size([8000, 11])

In [38]:
y_train_tensor

tensor([0., 0., 0.,  ..., 0., 0., 1.])

# Defining the Model

In [43]:
# Creating Model class
import torch
import torch.nn as nn

class Model(nn.Module):
    def __init__(self,num_features):
        super().__init__()

        self.network = nn.Sequential(
            # Input Layer -> Hidden Layer 1
            nn.Linear(num_features,6),
            # Activation Function
            nn.ReLU(),
            # Hidden Layer 1 -> Hidden Layer 2
            nn.Linear(6,3),
            # Activation Function
            nn.ReLU(),
            # Binary Classification
            nn.Linear(3, 1),
            nn.Sigmoid()

        )
    def forward(self,features):
        out= self.network(features)

        return out
    def loss_function(self,y_pred,y):
      # Clamping prediction to avoid log(0)
      epsilon=1e-7
      y_pred=torch.clamp(y_pred,epsilon,1-epsilon)
      # Calculate loss
      loss= -(y_train_tensor*torch.log(y_pred)+(1-y_train_tensor)*torch.log(1-y_pred))
      return loss.mean()

# Training Pipeline

In [40]:
!pip install torchinfo

In [41]:
# Create Model
model = Model(X_train_tensor.shape[1])
# Call model for forward pass
# model.forward(features)
model(X_train_tensor)

tensor([[0.4959],
        [0.4470],
        [0.4600],
        ...,
        [0.4132],
        [0.4777],
        [0.4638]], grad_fn=<SigmoidBackward0>)

In [42]:
# show model weights
from torchinfo import summary

summary(model, input_size=(8000, 11))

Layer (type:depth-idx)                   Output Shape              Param #
Model                                    [8000, 1]                 --
├─Sequential: 1-1                        [8000, 1]                 --
│    └─Linear: 2-1                       [8000, 6]                 72
│    └─ReLU: 2-2                         [8000, 6]                 --
│    └─Linear: 2-3                       [8000, 3]                 21
│    └─ReLU: 2-4                         [8000, 3]                 --
│    └─Linear: 2-5                       [8000, 1]                 4
│    └─Sigmoid: 2-6                      [8000, 1]                 --
Total params: 97
Trainable params: 97
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.78
Input size (MB): 0.35
Forward/backward pass size (MB): 0.64
Params size (MB): 0.00
Estimated Total Size (MB): 0.99

# Learning parameters

In [44]:
learning_rate= 0.1
epochs= 25

# Training Pipeline

In [46]:
# Create Model

model = Model(X_train_tensor.shape[1])

for epoch in range(epochs):
  # Forward pass
  y_pred = model(X_train_tensor)
  # Calculate loss
  loss = model.loss_function(y_pred,y_train_tensor)

  # backward pass
  loss.backward()

  # parameters update
  with torch.no_grad():
    for param in model.parameters():
      param -= learning_rate * param.grad
  # Reset gradients
  model.zero_grad()

  # Print loss in each epoch
  print(f'Epoch:{epoch+1},Loss:{loss.item()}')

Epoch:1,Loss:0.6990811228752136
Epoch:2,Loss:0.6895659565925598
Epoch:3,Loss:0.6805266737937927
Epoch:4,Loss:0.6719381809234619
Epoch:5,Loss:0.6637763381004333
Epoch:6,Loss:0.656020998954773
Epoch:7,Loss:0.6486520767211914
Epoch:8,Loss:0.6416512727737427
Epoch:9,Loss:0.6349973678588867
Epoch:10,Loss:0.6286723017692566
Epoch:11,Loss:0.622661292552948
Epoch:12,Loss:0.6169489622116089
Epoch:13,Loss:0.6115207076072693
Epoch:14,Loss:0.606357216835022
Epoch:15,Loss:0.6014440059661865
Epoch:16,Loss:0.5967706441879272
Epoch:17,Loss:0.5923241376876831
Epoch:18,Loss:0.5880945920944214
Epoch:19,Loss:0.5840719938278198
Epoch:20,Loss:0.5802465081214905
Epoch:21,Loss:0.576609194278717
Epoch:22,Loss:0.5731513500213623
Epoch:23,Loss:0.569863498210907
Epoch:24,Loss:0.5667348504066467
Epoch:25,Loss:0.5637576580047607
